# Train & Evaluation: D1 vs D2 vs D3 (Human Gold) — Colab

**Verification-Guided LLM Annotation Framework**

Pipeline toàn bộ trong 1 notebook:
1. Mount Drive + setup paths
2. Build D1_train / D2_train (loại D3)
3. Dataset statistics (Table 1)
4. Label quality D1/D2 vs D3 gold (Table 2) — **không cần train**
5. TF-IDF + SVM baseline (Table 3a)
6. PhoBERT-base / XLM-R-base fine-tune (Table 3b) — cần GPU
7. Tổng hợp kết quả

**Files cần có trên Drive** (`/content/drive/MyDrive/intent_kb/`):
```
data/raw/hasaki/hasaki_labelled_clean.json
data/hasaki_labelled_full_verified (1).json   ← hoặc data/raw/hasaki/hasaki_labelled_full_verified.json
data/gold/d3_human_gold.json                  ← 300 mẫu đã annotate (300/300)
data/datasets/d2_verified_approved.json       ← nếu đã có; nếu chưa cell §2 sẽ build
```

Chạy **tuần tự §1 → §7**.


## 1. Cài đặt packages

In [ ]:
!pip -q install scikit-learn pandas matplotlib
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip -q install transformers accelerate sentencepiece
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Mount Drive + paths

Đổi `DEFAULT_DRIVE_REPO` nếu folder trên Drive khác.


In [ ]:
import os, sys, json
from pathlib import Path
from collections import Counter

IN_COLAB = "google.colab" in sys.modules
DEFAULT_DRIVE_REPO = "/content/drive/MyDrive/intent_kb"

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    REPO_ROOT = Path(DEFAULT_DRIVE_REPO)
else:
    REPO_ROOT = Path(".").resolve()

RESULTS_DIR = REPO_ROOT / "experiments/evaluation/results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

D1_CLEAN_PATH  = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_clean.json"
D3_GOLD_PATH   = REPO_ROOT / "data/gold/d3_human_gold.json"
D2_VERIFIED_PATH = REPO_ROOT / "data/datasets/d2_verified_approved.json"
D1_TRAIN_PATH  = REPO_ROOT / "data/datasets/d1_annotator_train.json"
D2_TRAIN_PATH  = REPO_ROOT / "data/datasets/d2_verified_train.json"
D2_META_PATH   = REPO_ROOT / "data/datasets/d2_build_meta.json"

# Full verified (ưu tiên path mới; fallback path cũ)
D2_FULL_VERIFIED_PATH = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_full_verified.json"
_alt = REPO_ROOT / "data/hasaki_labelled_full_verified (1).json"
if not D2_FULL_VERIFIED_PATH.exists() and _alt.exists():
    D2_FULL_VERIFIED_PATH = _alt

print("REPO_ROOT:", REPO_ROOT)
for p in [D1_CLEAN_PATH, D3_GOLD_PATH, D2_VERIFIED_PATH, D2_FULL_VERIFIED_PATH]:
    print(f"  {'✅' if p.exists() else '❌'} {p.name}")


## 3. Helper functions (inline)

In [ ]:
# ── data helpers ─────────────────────────────────────────────────────────────

def load_results(path):
    with Path(path).open(encoding="utf-8") as f:
        doc = json.load(f)
    if isinstance(doc, dict) and "results" in doc:
        return list(doc["results"])
    if isinstance(doc, list):
        return doc
    raise ValueError(f"Unsupported JSON shape: {path}")


def intent_levels(row):
    intent = row.get("intent_3_level") or {}
    l3 = intent.get("level_3")
    if isinstance(l3, list):
        l3 = l3[0] if l3 else ""
    return (
        str(intent.get("level_1") or ""),
        str(intent.get("level_2") or ""),
        str(l3 or ""),
    )


def gold_levels(row):
    if row.get("gold_level_1"):
        return (str(row["gold_level_1"]), str(row.get("gold_level_2") or ""), str(row.get("gold_level_3") or ""))
    return intent_levels(row)


def rows_with_gold(rows):
    return [r for r in rows if all(gold_levels(r))]


def path_label(l1, l2, l3):
    return f"{l1}::{l2}::{l3}"


def parse_path(s):
    parts = s.split("::", 2)
    return tuple(parts + [""] * (3 - len(parts)))


# ── metrics ───────────────────────────────────────────────────────────────────

from sklearn.metrics import accuracy_score, f1_score, classification_report

_EMPTY_METRICS = {
    "l1_accuracy": 0, "l1_macro_f1": 0, "l1_weighted_f1": 0,
    "l2_accuracy": 0, "l2_macro_f1": 0, "l2_weighted_f1": 0,
    "l3_accuracy": 0, "l3_macro_f1": 0, "l3_weighted_f1": 0,
    "full_path_accuracy": 0, "n_samples": 0,
}


def compute_hierarchical_metrics(true_paths, pred_paths):
    """Per-level (L1/L2/L3) accuracy + Macro-F1 + Weighted-F1, plus full-path accuracy.

    Full-path accuracy = fraction of samples where L1, L2, L3 are ALL correct.
    """
    if not true_paths:
        return dict(_EMPTY_METRICS)

    levels_t = list(zip(*true_paths))   # [(l1...), (l2...), (l3...)]
    levels_p = list(zip(*pred_paths))

    out = {}
    for i, name in enumerate(("l1", "l2", "l3")):
        yt, yp = levels_t[i], levels_p[i]
        out[f"{name}_accuracy"]    = float(accuracy_score(yt, yp))
        out[f"{name}_macro_f1"]    = float(f1_score(yt, yp, average="macro",    zero_division=0))
        out[f"{name}_weighted_f1"] = float(f1_score(yt, yp, average="weighted", zero_division=0))

    full_t = [path_label(*p) for p in true_paths]
    full_p = [path_label(*p) for p in pred_paths]
    out["full_path_accuracy"] = float(accuracy_score(full_t, full_p))
    out["n_samples"] = len(true_paths)
    return out


def metrics_row(m, **extra):
    """Flatten metrics dict into a table row: per-level Macro/Weighted-F1 + full-path acc."""
    row = dict(extra)
    row.update({
        "L1 Acc":   m.get("l1_accuracy", 0),
        "L1 mF1":   m.get("l1_macro_f1", 0),
        "L1 wF1":   m.get("l1_weighted_f1", 0),
        "L2 Acc":   m.get("l2_accuracy", 0),
        "L2 mF1":   m.get("l2_macro_f1", 0),
        "L2 wF1":   m.get("l2_weighted_f1", 0),
        "L3 Acc":   m.get("l3_accuracy", 0),
        "L3 mF1":   m.get("l3_macro_f1", 0),
        "L3 wF1":   m.get("l3_weighted_f1", 0),
        "Full-path": m.get("full_path_accuracy", 0),
    })
    return row


def count_unique_levels(rows):
    if not rows: return 0, 0, 0
    return (
        len({intent_levels(r)[0] for r in rows}),
        len({intent_levels(r)[1] for r in rows}),
        len({intent_levels(r)[2] for r in rows}),
    )


print("Helpers ready ✅")


## 4. Build D2 + D1_train / D2_train

Nếu các file đã có sẵn trên Drive, cell này sẽ bỏ qua (SKIP).  
Thay đổi `FORCE_REBUILD = True` để rebuild lại từ đầu.


In [ ]:
import csv

FORCE_REBUILD = False   # True = rebuild dù file đã có

# ── build D2 từ full verified (nếu chưa có hoặc FORCE) ─────────────────────

if FORCE_REBUILD or not D2_VERIFIED_PATH.exists():
    if not D2_FULL_VERIFIED_PATH.exists():
        print("⚠️  Không tìm thấy full-verified file → dùng partial D2 nếu có")
    else:
        full_rows = load_results(D2_FULL_VERIFIED_PATH)
        approved = [
            {**r, "d2_source": "full_verified_approved"}
            for r in full_rows
            if r.get("qa_status") == "approved" and r.get("sample_id")
        ]
        d2_payload = {
            "dataset": "D2_verified_approved",
            "meta": {
                "mode": "full",
                "num_input_total": len(full_rows),
                "num_approved": len(approved),
                "verifier_decision_distribution": dict(Counter(
                    r.get("verifier_decision") for r in full_rows if r.get("verifier_decision")
                )),
            },
            "results": approved,
        }
        D2_VERIFIED_PATH.parent.mkdir(parents=True, exist_ok=True)
        with D2_VERIFIED_PATH.open("w", encoding="utf-8") as f:
            json.dump(d2_payload, f, ensure_ascii=False, indent=2)
        print(f"Built D2: {len(approved)} approved → {D2_VERIFIED_PATH.name}")
else:
    print(f"SKIP: D2 exists ({D2_VERIFIED_PATH.name})")

# ── build D1_train / D2_train (exclude D3) ──────────────────────────────────

if FORCE_REBUILD or not D1_TRAIN_PATH.exists() or not D2_TRAIN_PATH.exists():
    if not D3_GOLD_PATH.exists():
        raise FileNotFoundError(f"Không tìm thấy D3: {D3_GOLD_PATH}")

    d1_rows = load_results(D1_CLEAN_PATH)
    d2_rows = load_results(D2_VERIFIED_PATH)
    d3_rows = load_results(D3_GOLD_PATH)

    d3_ids  = {r["sample_id"] for r in d3_rows if r.get("sample_id")}
    d1_by_id = {r["sample_id"]: r for r in d1_rows if r.get("sample_id")}
    d2_by_id = {r["sample_id"]: r for r in d2_rows if r.get("sample_id")}

    matched_ids = sorted(set(d2_by_id.keys()) - d3_ids)

    def slim_row(row, source):
        l1, l2, l3 = intent_levels(row)
        return {
            "sample_id": row["sample_id"],
            "sentence": row.get("sentence", ""),
            "category": row.get("category", ""),
            "intent_3_level": {"level_1": l1, "level_2": l2, "level_3": l3},
            "confidence": row.get("confidence"),
            "qa_status": row.get("qa_status"),
            "label_source": source,
        }

    d1_train = [slim_row(d1_by_id[sid], "D1_annotator") for sid in matched_ids if sid in d1_by_id]
    d2_train = [slim_row(d2_by_id[sid], "D2_verified")  for sid in matched_ids if sid in d2_by_id]
    missing  = [sid for sid in matched_ids if sid not in d1_by_id]

    train_meta = {
        "d3_excluded_count": len(d3_ids),
        "d2_total": len(d2_by_id),
        "matched_train_ids": len(matched_ids),
        "d1_train_count": len(d1_train),
        "d2_train_count": len(d2_train),
        "missing_in_d1": missing,
    }

    D1_TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
    for path, dataset, rows in [
        (D1_TRAIN_PATH, "D1_annotator_train", d1_train),
        (D2_TRAIN_PATH, "D2_verified_train",  d2_train),
    ]:
        with path.open("w", encoding="utf-8") as f:
            json.dump({"dataset": dataset, "meta": train_meta, "results": rows}, f, ensure_ascii=False, indent=2)
    with (REPO_ROOT / "data/datasets/dataset_train_meta.json").open("w", encoding="utf-8") as f:
        json.dump(train_meta, f, ensure_ascii=False, indent=2)

    print(f"D3 excluded: {len(d3_ids)}")
    print(f"D1_train: {len(d1_train)}, D2_train: {len(d2_train)}")
    if missing:
        print(f"  ⚠️ {len(missing)} IDs in D2 but not in D1")
else:
    print(f"SKIP: train sets exist")
    d1_train = load_results(D1_TRAIN_PATH)
    d2_train = load_results(D2_TRAIN_PATH)


## 5. Load tất cả datasets

In [ ]:
import pandas as pd

d1_clean = load_results(D1_CLEAN_PATH)
d2       = load_results(D2_VERIFIED_PATH)
d3_all   = load_results(D3_GOLD_PATH)
d3_gold  = rows_with_gold(d3_all)

d1_train = load_results(D1_TRAIN_PATH) if D1_TRAIN_PATH.exists() else []
d2_train = load_results(D2_TRAIN_PATH) if D2_TRAIN_PATH.exists() else []

d3_ids   = {r["sample_id"] for r in d3_all}
train_ids = {r["sample_id"] for r in d1_train} | {r["sample_id"] for r in d2_train}

print(f"D1 clean : {len(d1_clean)}")
print(f"D2 verified: {len(d2)}")
print(f"D3 gold  : {len(d3_gold)}/300")
print(f"D1_train : {len(d1_train)}")
print(f"D2_train : {len(d2_train)}")

# Leakage check
leak = d3_ids & train_ids
assert len(leak) == 0, f"LEAKAGE: {len(leak)} D3 IDs in train! {sorted(leak)[:3]}"
print(f"✅ No leakage ({len(d3_ids)} D3 IDs checked vs {len(train_ids)} train IDs)")


## 6. Table 1 — Dataset statistics

In [ ]:
d3_display = d3_gold if d3_gold else d3_all
d3_label = f"D3 gold ({len(d3_gold)} annotated)" if d3_gold else f"D3 template ({len(d3_all)} pending)"

rows = [
    ("D1 full (annotator)",    d1_clean),
    ("D2 full (verified)",     d2),
    (d3_label,                 d3_display),
    ("D1_train (excl D3)",     d1_train),
    ("D2_train (excl D3)",     d2_train),
]

table1 = pd.DataFrame([
    {"Dataset": name, "N": len(r), "L1": count_unique_levels(r)[0],
     "L2": count_unique_levels(r)[1], "L3": count_unique_levels(r)[2]}
    for name, r in rows
])
display(table1.to_string(index=False))
table1.to_csv(RESULTS_DIR / "table1_dataset_stats.csv", index=False)
print(f"Saved → {RESULTS_DIR}/table1_dataset_stats.csv")


## 7. Table 2 — Label quality: D1 vs D3, D2 vs D3 (RQ1)

So sánh nhãn máy (D1/D2) với gold (D3) — **không train**.


In [ ]:
if not d3_gold:
    print("SKIP: D3 chưa annotate")
else:
    d1_by_id = {r["sample_id"]: r for r in d1_clean}
    d2_by_id = {r["sample_id"]: r for r in d2}

    def eval_vs_gold(name, by_id):
        true_paths, pred_paths = [], []
        for r in d3_gold:
            sid = r["sample_id"]
            if sid not in by_id:
                continue
            true_paths.append(gold_levels(r))
            pred_paths.append(intent_levels(by_id[sid]))
        if not true_paths:
            return {"dataset": name, "n_matched": 0}
        m = compute_hierarchical_metrics(true_paths, pred_paths)
        m["dataset"] = name
        m["n_matched"] = len(true_paths)
        return m

    m1 = eval_vs_gold("D1: GPT Annotator", d1_by_id)
    m2 = eval_vs_gold("D2: GPT Verified",  d2_by_id)

    table2 = pd.DataFrame([
        metrics_row(m, Dataset=m["dataset"], **{"N matched": m["n_matched"]})
        for m in [m1, m2]
    ])
    display(table2.round(4).to_string(index=False))
    table2.to_csv(RESULTS_DIR / "table2_label_quality.csv", index=False)

    delta_f1_l3 = m2.get("l3_macro_f1", 0) - m1.get("l3_macro_f1", 0)
    delta_full  = m2.get("full_path_accuracy", 0) - m1.get("full_path_accuracy", 0)
    print(f"\nD2 vs D1 improvement:  ΔL3 Macro-F1={delta_f1_l3:+.4f}  ΔFull-path={delta_full:+.4f}")


## 8. Table 3a — Downstream: TF-IDF + SVM (RQ3 baseline)

Train TF-IDF+LinearSVC trên D1_train và D2_train, test trên D3 gold.


In [ ]:
if not d3_gold or not d1_train or not d2_train:
    print("SKIP: Cần D3 gold, D1_train, D2_train")
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import Pipeline
    from sklearn.svm import LinearSVC

    def train_eval_svm(train_rows, name):
        X_tr = [r["sentence"] for r in train_rows]
        y_tr = [path_label(*intent_levels(r)) for r in train_rows]
        X_te = [r["sentence"] for r in d3_gold]
        y_true = [gold_levels(r) for r in d3_gold]

        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
            ("clf",   LinearSVC(class_weight="balanced", max_iter=5000, random_state=42)),
        ])
        pipe.fit(X_tr, y_tr)
        y_pred_str = pipe.predict(X_te)
        pred_paths = [parse_path(s) for s in y_pred_str]
        m = compute_hierarchical_metrics(y_true, pred_paths)
        m.update({"Train": name, "Model": "TF-IDF+SVM",
                  "n_train": len(train_rows), "n_test": len(d3_gold)})
        return m, pipe

    r1, pipe1 = train_eval_svm(d1_train, "D1_train")
    r2, pipe2 = train_eval_svm(d2_train, "D2_train")

    table3a = pd.DataFrame([
        metrics_row(r, Train=r["Train"], Model=r["Model"], **{"N train": r["n_train"]})
        for r in [r1, r2]
    ])
    display(table3a.round(4).to_string(index=False))
    table3a.to_csv(RESULTS_DIR / "table3a_tfidf_svm.csv", index=False)

    svm_results = [r1, r2]
    with (RESULTS_DIR / "tfidf_svm_metrics.json").open("w", encoding="utf-8") as f:
        json.dump([{k: v for k, v in r.items() if k not in ("n_samples",)} for r in svm_results],
                  f, ensure_ascii=False, indent=2)
    print(f"Saved → table3a_tfidf_svm.csv")


## 9. Table 3b — Downstream: PhoBERT-base / XLM-R-base (RQ3)

Cần **GPU**. Đổi `TRANSFORMER_MODEL` hoặc chạy cả hai bằng cách lặp.

| Model | Colab GPU | Thời gian (3 epoch) |
|-------|-----------|---------------------|
| `vinai/phobert-base` | T4 | ~15–20 phút |
| `xlm-roberta-base`   | T4 | ~20–25 phút |


In [ ]:
import torch

TRANSFORMER_MODEL = "vinai/phobert-base"   # hoặc "xlm-roberta-base"
EPOCHS      = 3
BATCH_SIZE  = 16
MAX_LENGTH  = 128
LEARNING_RATE = 2e-5

RUN_TRANSFORMER = True   # False = skip section này

if not RUN_TRANSFORMER:
    print("SKIP: đặt RUN_TRANSFORMER=True để chạy")
elif not d3_gold or not d1_train or not d2_train:
    print("SKIP: Cần D3 gold + train sets")
elif not torch.cuda.is_available():
    print("⚠️ Không có GPU — transformer sẽ rất chậm. Dùng TF-IDF kết quả §8 thay thế.")
else:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
    from torch.utils.data import Dataset

    class IntentDataset(Dataset):
        def __init__(self, texts, label_ids, tokenizer, max_length):
            self.enc = tokenizer(texts, truncation=True, padding="max_length",
                                 max_length=max_length, return_tensors="pt")
            self.label_ids = label_ids

        def __len__(self): return len(self.label_ids)

        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.label_ids[idx])
            return item


    def train_eval_transformer(train_rows, name, model_name):
        X_tr = [r["sentence"] for r in train_rows]
        y_tr = [path_label(*intent_levels(r)) for r in train_rows]
        X_te = [r["sentence"] for r in d3_gold]
        y_true_paths = [gold_levels(r) for r in d3_gold]
        y_true = [path_label(*p) for p in y_true_paths]

        all_labels = sorted(set(y_tr) | set(y_true))
        label2id = {l: i for i, l in enumerate(all_labels)}
        id2label  = {i: l for l, i in label2id.items()}

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=len(all_labels), id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True,
        )

        train_label_ids = [label2id[y] for y in y_tr]
        train_ds = IntentDataset(X_tr, train_label_ids, tokenizer, MAX_LENGTH)

        run_dir = RESULTS_DIR / "models" / f"{name}_{model_name.replace('/', '_')}"
        run_dir.mkdir(parents=True, exist_ok=True)

        args = TrainingArguments(
            output_dir=str(run_dir),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=0.01,
            logging_steps=50,
            save_strategy="no",
            report_to="none",
            seed=42,
            fp16=torch.cuda.is_available(),
        )

        trainer = Trainer(model=model, args=args, train_dataset=train_ds)
        print(f"Training {name} + {model_name} ({len(train_rows)} samples, {EPOCHS} epochs)...")
        trainer.train()

        model.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        test_enc = tokenizer(X_te, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
        test_enc = {k: v.to(device) for k, v in test_enc.items()}
        with torch.no_grad():
            pred_ids = model(**test_enc).logits.argmax(dim=-1).cpu().tolist()

        pred_paths = [parse_path(id2label.get(i, "::")) for i in pred_ids]
        hier = compute_hierarchical_metrics(y_true_paths, pred_paths)
        return {
            "train_set": name, "model_name": model_name,
            "n_train": len(train_rows), "n_test": len(d3_gold),
            "epochs": EPOCHS, **hier,
        }


    transformer_results = []
    for train_name, train_rows in [("D1_train", d1_train), ("D2_train", d2_train)]:
        r = train_eval_transformer(train_rows, train_name, TRANSFORMER_MODEL)
        transformer_results.append(r)
        print(f"{train_name}: L3 Macro-F1={r['l3_macro_f1']:.4f}  L3 wF1={r['l3_weighted_f1']:.4f}  Full-path={r['full_path_accuracy']:.4f}")

    table3b = pd.DataFrame([
        metrics_row(r, Train=r["train_set"], Model=r["model_name"].split("/")[-1], **{"N train": r["n_train"]})
        for r in transformer_results
    ])
    display(table3b.round(4).to_string(index=False))
    table3b.to_csv(RESULTS_DIR / f"table3b_{TRANSFORMER_MODEL.replace('/', '_')}.csv", index=False)

    safe = TRANSFORMER_MODEL.replace("/", "_")
    with (RESULTS_DIR / f"transformer_{safe}_metrics.json").open("w", encoding="utf-8") as f:
        json.dump(transformer_results, f, ensure_ascii=False, indent=2)
    print(f"Saved → table3b + transformer_{safe}_metrics.json")


## 10. Tổng hợp kết quả

In [ ]:
# 11. Ablation study — Effect of pipeline components
# Configs:
ablation_configs = [
    ("Abl-1_AnnotatorOnly", d1_train),                # annotator-only labels
    ("Abl-2_Annotator+Retrieval", d1_train),          # (placeholder) same labels; can augment with retrieval signals
    ("Abl-3_Annotator+Verifier", d2_train),           # verified labels (D2)
    ("Abl-4_FullPipeline_Guardrails", d2_train),      # same as D2 (guardrails already applied)
]

SEEDS = (42, 43, 44)

ablation_results = []

print("Running ablation configs:", [c[0] for c in ablation_configs])

# TF-IDF + LinearSVC ablation (fast)
for cfg_name, train_rows in ablation_configs:
    for seed in SEEDS:
        # Optionally augment train_rows per config (here kept simple)
        try:
            r, _ = train_eval_svm(train_rows, f"{cfg_name}_seed{seed}")
        except Exception as e:
            print("TFIDF training failed for", cfg_name, "seed", seed, "->", e)
            continue
        r.update({"ablation_cfg": cfg_name, "seed": seed, "method": "TFIDF+SVM"})
        ablation_results.append(r)

# Transformer ablation (if available / enabled)
if 'train_eval_transformer' in globals() and RUN_TRANSFORMER and torch.cuda.is_available():
    for cfg_name, train_rows in ablation_configs:
        for seed in SEEDS:
            try:
                # reuse train_eval_transformer (it uses TRANSFORMER_MODEL global)
                r = train_eval_transformer(train_rows, cfg_name, TRANSFORMER_MODEL)
            except Exception as e:
                print("Transformer training failed for", cfg_name, "seed", seed, "->", e)
                continue
            r.update({"ablation_cfg": cfg_name, "seed": seed, "method": f"TRANS_{TRANSFORMER_MODEL.split('/')[-1]}"})
            ablation_results.append(r)
else:
    print("Skipping transformer ablation: RUN_TRANSFORMER=False or no GPU or trainer not available")

# Aggregate results
import pandas as pd
if ablation_results:
    df_ab = pd.DataFrame(ablation_results)
    # keep relevant metrics and flatten
    cols = ["ablation_cfg", "method", "seed", "n_train", "n_test",
            "l1_macro_f1", "l1_weighted_f1", "l2_macro_f1", "l2_weighted_f1",
            "l3_macro_f1", "l3_weighted_f1", "full_path_accuracy"]
    exist_cols = [c for c in cols if c in df_ab.columns]
    df_ab = df_ab[exist_cols]
    out_path = RESULTS_DIR / "ablation_results.csv"
    df_ab.to_csv(out_path, index=False)
    print(f"Saved ablation results → {out_path}")
    display(df_ab.groupby(["ablation_cfg", "method"]).agg({
        "l3_macro_f1": ["mean", "std"],
        "l3_weighted_f1": ["mean", "std"],
        "full_path_accuracy": ["mean", "std"]
    }))
else:
    print("No ablation results collected.")


In [ ]:
print("=" * 70)
print("SUMMARY — Verification-Guided LLM Annotation Framework")
print("=" * 70)

# Table 2
if 'table2' in dir():
    print("\n[Table 2] Label Quality vs D3 Human Gold:")
    print(table2.round(4).to_string(index=False))

# Table 3a
if 'table3a' in dir():
    print("\n[Table 3a] TF-IDF + SVM:")
    print(table3a.round(4).to_string(index=False))

# Table 3b
if 'table3b' in dir():
    print(f"\n[Table 3b] Transformer ({TRANSFORMER_MODEL}):")
    print(table3b.round(4).to_string(index=False))

print(f"\nResults saved in: {RESULTS_DIR}")
import os
for f in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {f.name}")
